In [1]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import cv2
import numpy as np

In [3]:
DATA_DIR = "../data/processed/clahe/"
RAW_DATA_DIR = "../data/raw/"

print(os.listdir(DATA_DIR))

print(len(os.listdir(os.path.join(DATA_DIR, "Normal"))),
      len(os.listdir(os.path.join(DATA_DIR, "COVID"))),
      len(os.listdir(os.path.join(DATA_DIR, "Lung_Opacity"))),
      len(os.listdir(os.path.join(DATA_DIR, "Viral Pneumonia"))))

CLASSES = [
    "Normal",
    "COVID",
    "Lung_Opacity",
    "Viral Pneumonia"
]

['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia']
10192 3616 6012 1345


In [4]:
IMG_SIZE = 299

def load_image(image_path):
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    return image.astype(np.float32)


def load_mask(mask_path):
    mask = cv2.imread(mask_path)
    mask = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)
    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    return (mask > 0).astype(np.float32)

def apply_lung_mask(image, mask):
    return image * mask

# Feature bags

In [5]:
from skimage.feature import local_binary_pattern
import numpy as np

def extract_stats_on_mask(image, mask):
    binary = (mask > 0)
    pixels = image[binary]
    q1, med, q3 = np.percentile(pixels, [25, 50, 75])
    features = {
        "stat_mean":        pixels.mean(),
        "stat_std":         pixels.std(),
        "stat_min":         pixels.min(),
        "stat_max":         pixels.max(),
        "stat_q1":          q1,
        "stat_med":         med,
        "stat_q3":          q3,
        "stat_iqr":         q3 - q1,
        "stat_dark_pixels": float(np.sum(pixels < pixels.mean())),
    }
    return features

def extract_lbp_on_mask(image, mask, P=8, R=1, method="uniform"):
    lbp = local_binary_pattern(image, P, R, method)
    lbp_pixels = lbp[mask > 0]
    n_bins = P + 2
    hist, _ = np.histogram(lbp_pixels, bins=n_bins,
                           range=(0, n_bins), density=True)
    return {f"lbp_r{R}_b{i}": hist[i] for i in range(n_bins)}

def extract_lbp_zones_on_mask(image, mask, P=8, R=1, method="uniform"):
    """LBP séparé gauche / droite pour capturer l'asymétrie spatiale"""
    mid = image.shape[1] // 2
    features = {}
    for side, img_zone, mask_zone in [
        ("left",  image[:, :mid], mask[:, :mid]),
        ("right", image[:, mid:], mask[:, mid:]),
    ]:
        lbp = local_binary_pattern(img_zone, P, R, method)
        lbp_pixels = lbp[mask_zone > 0]
        n_bins = P + 2
        if len(lbp_pixels) == 0:  # masque vide sur cette zone
            hist = np.zeros(n_bins)
        else:
            hist, _ = np.histogram(lbp_pixels, bins=n_bins,
                                   range=(0, n_bins), density=True)
        features.update({f"lbp_{side}_r{R}_b{i}": hist[i] for i in range(n_bins)})
    return features

# Feature extraction pipeline

In [6]:
def extract_all_features(image, mask, feature_extractors):
    features = {}
    for extractor in feature_extractors:
        features.update(extractor(image, mask))
    return features


In [15]:
from tqdm import tqdm

def create_dataset(feature_extractors, output_path):
    rows = []

    for cls in CLASSES:
        cls_images_path = os.path.join(DATA_DIR, cls)
        cls_masks_path  = os.path.join(RAW_DATA_DIR, cls, "masks")

        cls_images = sorted(os.listdir(cls_images_path))
        cls_masks  = sorted(os.listdir(cls_masks_path))

        for img_name, mask_name in tqdm(zip(cls_images, cls_masks), desc=cls):
            image = load_image(os.path.join(cls_images_path, img_name))
            mask  = load_mask(os.path.join(cls_masks_path, mask_name))

            row = {"filename": img_name, "class": cls}
            row.update(extract_all_features(image, mask, feature_extractors))
            rows.append(row)

    pd.DataFrame(rows).to_csv(f"{output_path}")

# Train/Test pipeline

In [24]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score
from sklearn.utils.class_weight import compute_class_weight


def train_test(df, feature_cols, combo_name=""):
    META_COLS = ["filename", "class"]

    X = df[feature_cols].values
    y = df["class"].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    classes = np.unique(y_train)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
    class_weight_dict = dict(zip(classes, weights))

    svm = SVC(kernel="linear", C=1.0, gamma="scale",
              class_weight=class_weight_dict, random_state=42)
    svm.fit(X_train, y_train)

    y_pred = svm.predict(X_test)
    """
    print(f"\n{'='*55}")
    print(f"  {combo_name}  ({len(feature_cols)} features)")
    print(f"{'='*55}")
    print(classification_report(y_test, y_pred, target_names=classes))

    cm   = confusion_matrix(y_test, y_pred, labels=classes)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
    disp.plot(cmap="Blues")
    plt.title(f"Confusion Matrix — {combo_name}")
    plt.tight_layout()
    plt.show()
    """
    return f1_score(y_test, y_pred, average="macro")

# Ablation study on the feature bags

In [19]:
from itertools import combinations


META_COLS = ["filename", "class"]
results   = []

ALL_EXTRACTORS = [extract_stats_on_mask, extract_lbp_on_mask, extract_lbp_zones_on_mask]

all_combos = [
    combo
    for r in range(1, len(ALL_EXTRACTORS) + 1)
    for combo in combinations(ALL_EXTRACTORS, r)
]

i = 1
print("Exporting features....")
print()
for combo in all_combos:
    combo_name = "+".join(fn.__name__ for fn in combo)
    print(f"combo {i}/{len(all_combos)}: {combo_name}")
    FEATURE_DIR=os.path.join("..", "data", "features")
    os.makedirs(FEATURE_DIR, exist_ok=True)
    create_dataset(feature_extractors=combo, output_path=os.path.join(FEATURE_DIR, f"{combo_name}.csv"))
    df = pd.read_csv(f"../data/features/{combo_name}.csv", index_col=0)
    print(df.shape)
    print()
    i += 1

Exporting features....

combo 1/7: extract_stats_on_mask


Normal: 10192it [00:15, 648.85it/s]
COVID: 3616it [00:05, 659.28it/s]
Lung_Opacity: 6012it [00:09, 667.15it/s]
Viral Pneumonia: 1345it [00:02, 656.69it/s]


(21165, 11)

combo 2/7: extract_lbp_on_mask


Normal: 0it [00:00, ?it/s]d:\Documents\Alternance MLE\projet\2-DS\Analysis-of-COVID-19-Chest-X-rays\.venv\Lib\site-packages\skimage\feature\texture.py:385: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(
Normal: 10192it [02:17, 73.99it/s]
COVID: 3616it [00:55, 65.70it/s]
Lung_Opacity: 6012it [04:13, 23.68it/s]
Viral Pneumonia: 1345it [00:46, 29.09it/s]


(21165, 12)

combo 3/7: extract_lbp_zones_on_mask


...

(21165, 31)

combo 6/7: extract_lbp_on_mask+extract_lbp_zones_on_mask


Normal: 0it [00:00, ?it/s]d:\Documents\Alternance MLE\projet\2-DS\Analysis-of-COVID-19-Chest-X-rays\.venv\Lib\site-packages\skimage\feature\texture.py:385: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(
Normal: 10192it [03:53, 43.62it/s]
COVID: 3616it [01:50, 32.74it/s]
Lung_Opacity: 6012it [07:13, 13.87it/s]
Viral Pneumonia: 1345it [01:28, 15.17it/s]


(21165, 32)

combo 7/7: extract_stats_on_mask+extract_lbp_on_mask+extract_lbp_zones_on_mask


Normal: 0it [00:00, ?it/s]d:\Documents\Alternance MLE\projet\2-DS\Analysis-of-COVID-19-Chest-X-rays\.venv\Lib\site-packages\skimage\feature\texture.py:385: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(
Normal: 10192it [11:38, 14.59it/s]
COVID: 3616it [04:41, 12.86it/s]
Lung_Opacity: 6012it [06:36, 15.17it/s]
Viral Pneumonia: 1345it [00:47, 28.44it/s]


(21165, 41)



In [25]:
results = []

i = 1
print("Training....")
print()

for combo in all_combos:
    combo_name = "+".join(fn.__name__ for fn in combo)
    print(f"combo {i}/{len(all_combos)}: {combo_name}")
    df = pd.read_csv(f"../data/features/{combo_name}.csv", index_col=0)
    feature_cols = [c for c in df.columns if c not in META_COLS and not c.startswith("Unnamed")]
    macro_f1     = train_test(df, feature_cols, combo_name=combo_name)

    results.append({
        "combination" : combo_name,
        "n_features"  : len(feature_cols),
        "macro_f1"    : round(macro_f1, 4),
    })
    i += 1

# ── Résumé final ──────────────────────────────────────────────────────────────

results_df = pd.DataFrame(results).sort_values("macro_f1", ascending=False)
print("\n" + "="*55)
print("  CLASSEMENT FINAL")
print("="*55)
print(results_df.to_string(index=False))

Training....

combo 1/7: extract_stats_on_mask
combo 2/7: extract_lbp_on_mask
combo 3/7: extract_lbp_zones_on_mask
combo 4/7: extract_stats_on_mask+extract_lbp_on_mask
combo 5/7: extract_stats_on_mask+extract_lbp_zones_on_mask
combo 6/7: extract_lbp_on_mask+extract_lbp_zones_on_mask
combo 7/7: extract_stats_on_mask+extract_lbp_on_mask+extract_lbp_zones_on_mask

  CLASSEMENT FINAL
                                                        combination  n_features  macro_f1
extract_stats_on_mask+extract_lbp_on_mask+extract_lbp_zones_on_mask          39    0.5742
                    extract_stats_on_mask+extract_lbp_zones_on_mask          29    0.5601
                          extract_stats_on_mask+extract_lbp_on_mask          19    0.5507
                      extract_lbp_on_mask+extract_lbp_zones_on_mask          30    0.5150
                                          extract_lbp_zones_on_mask          20    0.5097
                                                extract_lbp_on_mask          